# Color Vision Deficiency (CVD) Accessibility

Ensure your visualizations are accessible to people with color blindness.

## What You'll Learn

1. Color vision deficiency types and statistics
2. CVD simulation using colorspacious library
3. Testing colormaps under CVD conditions
4. Accessibility recommendations
5. Design guidelines for inclusive visualization

**Prerequisites**: Notebook #4 (Colormaps), colorspacious library

**Estimated time**: ~20 minutes

**Critical**: This addresses accessibility - 8% of males have some form of CVD!

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import complexplorer as cp

# Check for colorspacious (needed for CVD simulation)
try:
    from colorspacious import cspace_convert
    HAS_COLORSPACIOUS = True
    print("✓ colorspacious is available")
except ImportError:
    print("⚠️ colorspacious not installed")
    print("   Install with: pip install colorspacious")
    HAS_COLORSPACIOUS = False

# Standard function
def f(z):
    return (z - 1) / (z**2 + z + 1)

domain = cp.Disk(radius=2)

## Introduction to Color Vision Deficiency

### Statistics
- **Protanopia** (red-blind): ~1% of males
- **Deuteranopia** (green-blind): ~1% of males  
- **Tritanopia** (blue-blind): ~0.001% (very rare)
- **Total**: ~8% of males, ~0.5% of females have some form

### Why It Matters
Scientific visualizations should be accessible to all viewers!

## CVD Simulation Function

In [ ]:
def simulate_cvd(rgb_image, cvd_type='protanomaly', severity=100):
    """Simulate color vision deficiency.
    
    Parameters
    ----------
    rgb_image : ndarray, shape (H, W, 3)
        RGB image with values in [0, 1]
    cvd_type : str
        'protanomaly' (red), 'deuteranomaly' (green), 'tritanomaly' (blue)
    severity : int
        0-100, where 100 = complete deficiency (-opia)
    
    Returns
    -------
    ndarray : Simulated RGB image
    """
    if not HAS_COLORSPACIOUS:
        print("colorspacious required for CVD simulation")
        return rgb_image
    
    # Flatten to (N, 3) array
    original_shape = rgb_image.shape
    rgb_flat = rgb_image.reshape(-1, 3)
    
    # Convert to CVD space and back
    cvd_space = {"name": "sRGB1+CVD", 
                 "cvd_type": cvd_type, 
                 "severity": severity}
    
    rgb_cvd = cspace_convert(rgb_flat, cvd_space, "sRGB1")
    
    # Clip to valid range and reshape
    rgb_cvd = np.clip(rgb_cvd, 0, 1)
    return rgb_cvd.reshape(original_shape)

if HAS_COLORSPACIOUS:
    print("✓ CVD simulation function ready")

## Test Single Colormap Under All CVD Types

In [ ]:
if HAS_COLORSPACIOUS:
    # Choose a colormap to test
    cmap = cp.Phase(phase_sectors=6, auto_scale_r=True)
    
    # Generate the plot data
    z = domain.mesh(500)
    mask = domain.outmask(500)
    fz = f(z)
    rgb = cmap.rgb(fz, outmask=mask)
    
    # Simulate different CVD types
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Normal vision
    axes[0,0].imshow(rgb, origin='lower')
    axes[0,0].set_title("Normal Vision", fontsize=12)
    axes[0,0].axis('off')
    
    # Protanopia (red-blind)
    rgb_prot = simulate_cvd(rgb, 'protanomaly', 100)
    axes[0,1].imshow(rgb_prot, origin='lower')
    axes[0,1].set_title("Protanopia (Red-Blind)", fontsize=12)
    axes[0,1].axis('off')
    
    # Deuteranopia (green-blind)
    rgb_deut = simulate_cvd(rgb, 'deuteranomaly', 100)
    axes[0,2].imshow(rgb_deut, origin='lower')
    axes[0,2].set_title("Deuteranopia (Green-Blind)", fontsize=12)
    axes[0,2].axis('off')
    
    # Tritanopia (blue-blind)
    rgb_trit = simulate_cvd(rgb, 'tritanomaly', 100)
    axes[1,0].imshow(rgb_trit, origin='lower')
    axes[1,0].set_title("Tritanopia (Blue-Blind)", fontsize=12)
    axes[1,0].axis('off')
    
    # Grayscale
    rgb_gray = np.dot(rgb[...,:3], [0.299, 0.587, 0.114])
    rgb_gray = np.stack([rgb_gray]*3, axis=-1)
    axes[1,1].imshow(rgb_gray, origin='lower')
    axes[1,1].set_title("Grayscale Conversion", fontsize=12)
    axes[1,1].axis('off')
    
    axes[1,2].axis('off')
    
    fig.suptitle(f"Traditional Phase Under Different CVD Conditions", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print("\n⚠️ Notice how traditional Phase loses structure under protanopia/deuteranopia!")
else:
    print("Install colorspacious to see CVD simulation")

## Compare Multiple Colormaps Under CVD

In [ ]:
if HAS_COLORSPACIOUS:
    # Test these colormaps for accessibility
    test_colormaps = [
        ('Phase', cp.Phase(phase_sectors=6, auto_scale_r=True)),
        ('PerceptualPastel', cp.PerceptualPastel(phase_sectors=6, auto_scale_r=True)),
        ('CubehelixPhase', cp.CubehelixPhase(phase_sectors=6, auto_scale_r=True)),
        ('InkPaper', cp.InkPaper(phase_strength=0.08)),
    ]
    
    fig, axes = plt.subplots(len(test_colormaps), 3, figsize=(12, 12))
    
    for row, (name, cmap) in enumerate(test_colormaps):
        # Generate RGB
        rgb = cmap.rgb(fz, outmask=mask)
        
        # Normal
        axes[row,0].imshow(rgb, origin='lower')
        axes[row,0].set_title(f"{name} (Normal)" if row==0 else "", fontsize=10)
        axes[row,0].set_ylabel(name, fontsize=11)
        axes[row,0].axis('off')
        
        # Protanopia
        rgb_cvd = simulate_cvd(rgb, 'protanomaly', 100)
        axes[row,1].imshow(rgb_cvd, origin='lower')
        axes[row,1].set_title("Protanopia" if row==0 else "", fontsize=10)
        axes[row,1].axis('off')
        
        # Grayscale
        rgb_gray = np.dot(rgb[...,:3], [0.299, 0.587, 0.114])
        rgb_gray = np.stack([rgb_gray]*3, axis=-1)
        axes[row,2].imshow(rgb_gray, origin='lower')
        axes[row,2].set_title("Grayscale" if row==0 else "", fontsize=10)
        axes[row,2].axis('off')
    
    fig.suptitle("Colormap Comparison Under CVD Conditions", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print("\n✓ CubehelixPhase and PerceptualPastel maintain structure better!")
else:
    print("Install colorspacious to see comparison")

## Accessibility Rankings

Based on CVD simulation tests:

### Best for CVD (Highly Recommended)
✅ **CubehelixPhase**: Designed for grayscale, excellent CVD performance
✅ **PerceptualPastel**: OkLCh perceptual uniformity helps
✅ **InkPaper**: Minimal color, relies on brightness

### Good (With Caveats)
⚠️ **Isoluminant**: Constant lightness can be problematic if not enough brightness contrast
⚠️ **AnalogousWedge**: Depends on hue range chosen

### Problematic
❌ **Traditional Phase**: Red-green confusion in protanopia/deuteranopia
❌ **Chessboard/PolarChessboard**: Color-dependent grid patterns

## Design Guidelines for Accessible Figures

### 1. Use Luminance Contrast
Don't rely on hue alone - vary brightness too!

### 2. Supplement with Patterns
Use `phase_sectors` parameter to add angular wedges:
```python
cmap = cp.PerceptualPastel(phase_sectors=6, auto_scale_r=True)
```

### 3. Test in Grayscale
Convert your figure to grayscale - does it still make sense?

### 4. Avoid Red-Green Combinations
Most common CVD types confuse red and green.

### 5. Provide Alternative Views
Include both colorful and grayscale versions in papers.

## Recommended Workflow

When creating figures for publication:

1. **Choose accessible colormap**: CubehelixPhase, PerceptualPastel, or InkPaper
2. **Add structural patterns**: Use `phase_sectors` and `auto_scale_r`
3. **Test CVD**: Use this notebook's simulation
4. **Check grayscale**: Ensure structure remains visible
5. **Consider supplements**: Provide grayscale version in appendix

## Summary

✅ CVD affects ~8% of males - accessibility matters!
✅ Simulation with colorspacious library
✅ CubehelixPhase and PerceptualPastel are most accessible
✅ Always test grayscale conversion
✅ Supplement color with patterns (phase_sectors)

## References

- Viénot et al. (1999): "Digital video colourmaps for checking the legibility of displays by dichromats"
- [colorspacious documentation](https://colorspacious.readthedocs.io/)
- [Color Universal Design](https://jfly.uni-koeln.de/color/)

## What's Next?

- **Notebook #6**: Riemann Sphere deep dive
- **Notebook #7**: Modulus scaling modes

Create inclusive visualizations! 🌈♿